# Task 7: pairwise judging with a Kaggle model-proxy judge

Create this notebook as a **Kaggle Benchmarks utility notebook** (Benchmarks -> create a task/utility notebook)
so `kaggle_benchmarks` model calls are available, with **Internet: on**. No GPU is needed.

The notebook only *reads saved pairwise JSONL files* (built by `python -m alignment.pairs` and committed under
`outputs/eval/`); it never regenerates model outputs. If you attached the generation-output dataset as an input
instead, set `PAIRS_DIR` to its path. Judgments are written to `/kaggle/working/eval` after every batch of 5
pairs, so re-running a cell (or a new session with the previous version's output attached) resumes without
re-spending calls.

In [ ]:
REPO_URL = "https://github.com/AyanAnees326/small-model-alignment.git"
BRANCH = "testing"
WORKDIR = "/kaggle/working/small-model-alignment"
JUDGE_MODEL = "google/gemini-3.6-flash"

!git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORKDIR} || git -C {WORKDIR} pull

In [ ]:
import glob, json, shutil, sys, time
from pathlib import Path

PAIRS_DIR = Path(WORKDIR) / "outputs" / "eval"   # or Path("/kaggle/input/<generation-output-dataset>")
OUT_DIR = Path("/kaggle/working/eval")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# resume across sessions: pick up judgments committed to the repo or from an attached earlier notebook version
for path in glob.glob(f"{PAIRS_DIR}/*_judged.jsonl") + glob.glob("/kaggle/input/**/*_judged.jsonl", recursive=True):
    if not (OUT_DIR / Path(path).name).exists():
        shutil.copy(path, OUT_DIR)
        print("resuming from", path)

pair_files = sorted(PAIRS_DIR.glob("*_pairs.jsonl"))
print([p.name for p in pair_files])

In [ ]:
sys.path.insert(0, str(Path(WORKDIR) / "scripts"))
import kaggle_benchmarks as kbench
import judge_pairwise_kaggle as J

call, judge_name = J.kaggle_judge(JUDGE_MODEL)
print("judge:", judge_name)
print(call('Reply with JSON only: {"judgments": [{"pair": 1, "reason": "test", "verdict": "A"}]}', "judge-smoke-0"))
time.sleep(5)  # keep >= 5 s between proxy requests

Equivalent CLI per file: `python scripts/judge_pairwise_kaggle.py --pairs_path ... --output_path ... --judge_model google/gemini-3.6-flash --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments`. It is called in-process here so the benchmark model handle is shared.

In [ ]:
summaries = {}
for pairs_path in pair_files:
    out = OUT_DIR / pairs_path.name.replace("_pairs.jsonl", "_judged.jsonl")
    summaries[pairs_path.stem] = J.judge_file(pairs_path, out, call, judge_name, batch_size=5, sleep_seconds=5,
                                             max_retries=3, reuse=True)

In [ ]:
cols = ["n_pairs", "win", "loss", "tie", "unknown", "failure", "win_rate_ties_half", "win_rate_excluding_ties",
        "retried_pairs", "unresolved_failure_batches"]
for name, s in summaries.items():
    print(f"{name:42s}", {c: s[c] for c in cols})
shutil.make_archive("/kaggle/working/judgments", "zip", OUT_DIR)

Save a notebook version when every file reports `failure == 0` (re-run the judging cell if not; completed pairs
are skipped). Then copy `/kaggle/working/eval/*` (or `judgments.zip`) into `outputs/eval/` of the repository and
commit it. The report tables are generated from the `*_summary.json` files.